## Simplified self-attention 

### Inputs embedings

In [2]:
import torch
inputs = torch.tensor(
[[0.43, 0.15, 0.89], # Your (x^1)
[0.55, 0.87, 0.66], # journey (x^2)
[0.57, 0.85, 0.64], # starts (x^3)
[0.22, 0.58, 0.33], # with (x^4)
[0.77, 0.25, 0.10], # one (x^5)
[0.05, 0.80, 0.55]] # step (x^6)
)

### calculate the intermediate attention scores

In [ ]:
query = inputs[1] # "journey" (x^2)
attn_scores_2 = torch.empty(inputs.shape[0])# attention scores for "journey"
for i, x_i in enumerate(inputs):
    # Compute the dot product between x_i and the query vector
    attn_scores_2[i] = torch.dot(x_i, query)
print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


### Normalize

In [ ]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum() # Normalize the scores
print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


### Softmax function

In [6]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)
attn_weights_2_naive = softmax_naive(attn_scores_2)
print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


### PyTorch implementation of softmax,

In [7]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


In [10]:
# Selecciona el segundo vector de inputs como query de referencia
query = inputs[1]

# Inicializa el vector de contexto con ceros, manteniendo las mismas dimensiones que query
context_vec_2 = torch.zeros(query.shape)

# Bucle que itera sobre cada vector de entrada junto con su índice
for i,x_i in enumerate(inputs):
    # Suma ponderada: multiplica cada vector x_i por su peso de atención correspondiente
    # attn_weights_2[i] es un escalar (peso) que determina cuánto contribuye x_i
    # Esta operación implementa: context_vec += peso_i * vector_i
    context_vec_2 += attn_weights_2[i]*x_i

# Imprime el vector de contexto final calculado
# Este representa una combinación ponderada de todos los vectores de entrada
print(context_vec_2)
print("\nVERIFICACIÓN - Cálculo elemento por elemento:")
for j in range(len(query)):
    suma_elemento = sum(attn_weights_2[i] * inputs[i][j] for i in range(len(inputs)))
    print(f"z^(2)[{j}] = ", end="")
    terminos = [f"{attn_weights_2[i]:.3f}×{inputs[i][j]:.3f}" for i in range(len(inputs))]
    print(" + ".join(terminos) + f" = {suma_elemento:.4f}")


tensor([0.4419, 0.6515, 0.5683])

VERIFICACIÓN - Cálculo elemento por elemento:
z^(2)[0] = 0.139×0.430 + 0.238×0.550 + 0.233×0.570 + 0.124×0.220 + 0.108×0.770 + 0.158×0.050 = 0.4419
z^(2)[1] = 0.139×0.150 + 0.238×0.870 + 0.233×0.850 + 0.124×0.580 + 0.108×0.250 + 0.158×0.800 = 0.6515
z^(2)[2] = 0.139×0.890 + 0.238×0.660 + 0.233×0.640 + 0.124×0.330 + 0.108×0.100 + 0.158×0.550 = 0.5683


In [11]:
attn_scores = torch.empty(6, 6)
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


### matrix multiplication

In [ ]:
attn_scores = inputs @ inputs.T 
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [13]:
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [14]:
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


## self-attention with trainable weights